# 7a — The lay of the land

Seven chapters in — capture, the two-tier cache, the IR, rewriting,
lowering — this interlude surveys the language itself: what Python the DSL
accepts today, what it refuses and why, what "type inference" means here,
and how we compare to numba and the kernel DSLs.

It is the first of a recurring series. After later rule-pack milestones, a
short `chNNa` interlude records the **deltas** — what just became possible
— rather than restating this baseline. The cells here run live against the
current rule pack (the supported list is printed *from* it), so if an
output ever disagrees with the prose, a later chapter has already moved
the frontier and the next interlude names the change.

Statuses used throughout:

- ✅ **supported today** — lowers, typed, tested.
- 🔧 **a registration away** — an entry in a rule pack; zero new machinery.
- 🏗️ **needs planned machinery** — named chapter; real design content.
- 🚫 **out by design** — with the reason.


In [1]:
# The language IS data: the supported surface, printed from the rule pack.
from pdum.dsl.stdlib.base_lang import _CASTS, LOWER_RULES

GLOSS = {
    "Expr": "docstrings / stray constants (ignored)",
    "Assign": "single-name assignment (SSA rebinding allowed)",
    "Return": "return <expr> (must be reached)",
    "Constant": "int / float / bool literals (honest types: 1 is i64)",
    "Name": "parameters, locals, captures",
    "BinOp": "+ - * / % **  (STRICT: operands must share a type)",
    "UnaryOp": "unary -",
    "Compare": "single < > <= >= == !=",
    "IfExp": "a if cond else b  (branch types must match)",
    "Call": "float()/int()/bool() casts · captured-kernel calls (inlined)",
}
print("the base pack accepts", len(LOWER_RULES), "syntax forms:\n")
for node_type in LOWER_RULES:
    print(f"  ✅ ast.{node_type.__name__:<10} — {GLOSS.get(node_type.__name__, '')}")
print("\n  casts:", ", ".join(sorted(_CASTS)))

the base pack accepts 21 syntax forms:

  ✅ ast.Expr       — docstrings / stray constants (ignored)
  ✅ ast.Assign     — single-name assignment (SSA rebinding allowed)
  ✅ ast.AugAssign  — 
  ✅ ast.Return     — return <expr> (must be reached)
  ✅ ast.If         — 
  ✅ ast.For        — 
  ✅ ast.While      — 
  ✅ ast.Break      — 
  ✅ ast.Continue   — 
  ✅ ast.Pass       — 
  ✅ ast.Constant   — int / float / bool literals (honest types: 1 is i64)
  ✅ ast.Name       — parameters, locals, captures
  ✅ ast.BinOp      — + - * / % **  (STRICT: operands must share a type)
  ✅ ast.UnaryOp    — unary -
  ✅ ast.BoolOp     — 
  ✅ ast.Compare    — single < > <= >= == !=
  ✅ ast.IfExp      — a if cond else b  (branch types must match)
  ✅ ast.Tuple      — 
  ✅ ast.Subscript  — 
  ✅ ast.Attribute  — 
  ✅ ast.Call       — float()/int()/bool() casts · captured-kernel calls (inlined)

  casts: bool, float, int


In [2]:
# What the language REFUSES, it refuses loudly — the errors are the index.
# Every function below is real source in this cell; each shows its actual error.
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.stdlib.base_lang import LOWER_RULES


@jit()
def wants_tuple(x):
    return (x, x)  # tuples: a dialect decision (vec? record?) — registration away


@jit()
def wants_attr(x):
    return x.real  # attributes: records/swizzles land with their dialects


@jit()
def wants_index(a):
    return a[10]  # subscripts: the arrays chapter


@jit()
def wants_for(x):
    total = x
    for i in range(4):  # statements need region lowering + the strict join policy
        total = total + x
    return total


@jit()
def wants_boolop(x):
    return x > 0.0 and x < 1.0  # and/or: registration + a short-circuit decision


@jit()
def wants_augassign(x):
    x += 1.0  # pure sugar for x = x + 1.0
    return x


# As of chapter 7 every one of these refuses. A ✅ line means a later
# chapter's rule pack widened the base language — the next interlude has it.
for h in (wants_tuple, wants_attr, wants_index, wants_for, wants_boolop, wants_augassign):
    try:
        lower_handle(h, LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
        print(f"  ✅ {h.fntype.template.label}: lowers now")
    except Exception as e:
        print(f"  {type(e).__name__:12s} {e}")

  ✅ wants_tuple: lowers now
  MissingRule  attribute 'real' needs a Record-typed value [/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_86664/3146915779.py:17:11]
  TypeError    cannot extract index 10 from f64 [/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_86664/3146915779.py:22:11]
  ✅ wants_for: lowers now
  ✅ wants_boolop: lowers now
  ✅ wants_augassign: lowers now


## Case study: `x = a[10].b`

The question this example really asks: *what does type inference have to do
here?* Answer: **nothing exotic** — at specialization time `a` arrives with
a concrete type, and every step is one local op rule reading it. The gap
today is vocabulary (no subscript rule, no index op), not machinery:

In [3]:
# Today: refused, with the exact gap named.
@jit()
def foo(a):
    x = a[10].b
    return x


try:
    lower_handle(foo, LOWER_RULES, CORE_OPS, arg_types=(T.f64,))
except Exception as e:
    print(type(e).__name__, "-", e)

TypeError - cannot extract index 10 from f64 [/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_86664/820589655.py:4:8]


In [4]:
# The same computation, hand-built — a five-line TOY dialect adds the index
# op with its type rule, and the types chain with zero inference machinery:
from pdum.dsl.kernel.ir import Builder, Region
from pdum.dsl.kernel.ops import OpDef
from pdum.dsl.kernel.printer import print_program

P = T.Record("P", (("b", T.f32), ("pad", T.f32)))
ArrP = T.Array(P, 1, "C", "<", True)


def _index_rule(args, attrs, regions):
    return args[0].dtype  # Array(dtype=...) -> its element type. That's it.


TOY = dict(CORE_OPS)
TOY["tensor.index"] = OpDef("tensor.index", _index_rule)

tb = Builder(TOY)
a = tb.emit("core.env", type=ArrP, slot=(0,))
ten = tb.emit("core.const", type=T.i64, value=10)
elem = tb.emit("tensor.index", a, ten)  # : P        (rule read Array.dtype)
x = tb.emit("core.field", elem, name="b")  # : f32   (rule read Record.fields)
print(print_program(Region(body=(tb.emit("core.yield", x),)), name="foo"))
print()
print("a[10]   :", elem.type)
print("a[10].b :", x.type, "  <- two local rules; 'inference' never appears")

foo() {
  %0 = core.env {slot = (0,)} : array<P{b: f32, pad: f32},1d,C>
  %1 = core.const {value = 10} : i64
  %2 = tensor.index %0, %1 : P{b: f32, pad: f32}
  %3 = core.field %2 {name = 'b'} : f32
  core.yield %3
}

a[10]   : P{b: f32, pad: f32}
a[10].b : f32   <- two local rules; 'inference' never appears


## What "type inference" is here — and is it sufficient?

**Forward-only typing inside the fused lowering pass.** Types enter through
exactly three doors — argument types at specialization, capture types via
`typeof`, literal types via `typeof` — and propagate through per-op
`type_rule`s. No unification, no fixpoint, no backward flow.

Why that is *sufficient* rather than immature: **specialization concretizes
everything at entry** (the Julia/numba insight — inference with concrete
argument types is just forward evaluation over types), and the strict-core
decision extends to joins: when `if`/`for` *statements* land, a name bound
in both branches must have the SAME type (loud otherwise), and `core.for`
carries are explicitly typed. Same-type-or-loud at every join is exactly
what eliminates numba's `typeinfer.py` fixpoint machinery and phi-node
unification — the single most complex component of that compiler, deleted
by a language decision.

**How dialects strengthen it** — four doors, no fifth:
1. **op type rules** — a dialect's ops carry their own typing (the toy
   `tensor.index` above);
2. **lower_ast rules** — a dialect decides what syntax *means* (and whether
   it auto-inserts casts);
3. **overloads** (step 8) — type-driven selection per target;
4. **post-IR analysis passes** — shapes, units, ranges, solver-backed
   checks (§12 of the architecture) run over the frozen Region, where
   multi-pass is cheap.


## The cross-library matrix

Construct-level comparison against the systems that define user expectations
(sources: current official docs, empirically spot-checked against numba
0.66 — full surveys in `docs/design/research/R10`/`R11`). Our column uses the
four statuses; the point of each row is the *decision* it surfaces, not the
checkmark.

| construct | **pdum.dsl** | numba | Triton | Taichi | cupy.jit |
|---|---|---|---|---|---|
| ternary `a if c else b` | ✅ | ✅ | ✅ | ✅ | ✅ |
| aug-assign `x += 1` | 🔧 pure desugar | ✅ | ✅ | ✅ (atomic on fields) | ✅ |
| chained `0 < x < 1` | 🔧 pure desugar | ✅ | scalars only | ✅ | 🚫 |
| `and` / `or` | 🔧 + one policy call | ✅ short-circuit | scalars only | bitwise, no short-circuit | ✅ |
| tuples + unpacking | 🔧 (`Tuple` type exists; ops are a dialect addition like `tensor.index` above) | ✅ | ✅ | partial | ✅ |
| `if`/`else` *statement* | 🏗️ ch12 — region lowering + strict joins | ✅ | ✅ | ✅ | ✅ |
| `for` + `range` | 🏗️ ch12 (`core.for` op already exists) | ✅ | `range` only | ✅ auto-parallelized | `range` only |
| early / multiple `return` | 🚫 settled 2026-07-12: one return, at the tail | ✅ must unify | top-level only | 🚫 one return, at the bottom | ✅ |
| `while` | 🏗️ needs a fourth region op; `for` covers kernel targets first | ✅ | ✅ | serial only | ✅ |
| `a[i]` subscript | 🏗️ arrays chapter | ✅ | load-only | read+write | element only |
| `.field` attribute | 🔧 once `@record` kinds land (`core.field` works today) | `@jitclass` | 🚫 | ✅ structs | 🚫 |
| closures / nested defs | ✅ **the founding abstraction** — captured kernels inline, typed env in the key | partial: cannot be returned | 🚫 module-level only | 🚫 module-level only | 🚫 |
| global reads | 🚫 loud `NameFateError` | frozen silently at compile | constexpr-only, else error | frozen silently | frozen **silently**, never invalidated |
| runtime recursion | 🚫 (inlining model) | ✅ limited | 🚫 | 🚫 | 🚫 |
| comprehensions | 🚫 in-kernel — host Python at capture covers the compile-time uses | partial | tuple-only | compile-time only | 🚫 |
| `try` / `with` / `match` / `yield` | 🚫 by design — no exception objects or effects inside kernels | partial / objmode | 🚫 | 🚫 | 🚫 |
| kwargs / defaults | 🏗️ dispatch layer (step 8) | ✅ | calls only | partial | 🚫 |


### What the matrix decides

**The consensus core is tiny.** Arithmetic, `if`, `for`+`range`, and
calls-that-inline is the entire intersection of the three GPU DSLs. We are
four *registrations* (aug-assign, bool ops, chained compares, tuples) and
one *machinery step* (ch12 statement lowering: `if`/`for`/early-return over
the region ops that already exist) away from it. That ordering goes into
the plan.

**Three rows are policy, not capability** — and land in *dialects*, never
the kernel:

1. `and`/`or` — the field disagrees (numba short-circuits, Taichi lowers to
   bitwise, Triton refuses tensors): the base pack will pick short-circuit
   via `core.if` and say so; a SIMD dialect may pick bitwise.
2. early return — numba unifies return paths; Taichi refuses all but one.
   **Settled at this walkthrough (2026-07-12): we take the strict end — one
   `return`, at the tail; `core.yield` IS the return.** No return-path
   unification machinery, ever (010 ledger).
3. globals — everyone else freezes silently (cupy never even invalidates);
   we keep the loud `NameFateError`. A silent snapshot is the staleness bug
   this project exists to kill: **capture it or pass it.**

**Specialization is where we quietly win.** Triton's `tl.constexpr` +
`do_not_specialize`, Taichi's `ti.template()` + `ti.static`, cupy's
implicit per-signature keying — three ad-hoc syntaxes for one need — all
map onto machinery this design already has: per-kind fingerprinting
(value- vs type-specialization as a `ValueKind` decision), the config
bracket's strip → value → type pipeline, and plain host Python at capture
time standing in for `ti.static`.


## Classes and dataclasses as structs

numba's `@jitclass` declares typed fields and compiles methods; its modern
successor `structref` is the acknowledged precedent (research: R8, R10). Our
equivalents are **already designed, scheduled at the surfaces chapter**:

- a `@record`-style decorator (surface C) registers a frozen dataclass as a
  `ValueKind`: `typeof` → the `Record` type below, `flatten` → its fields;
- field access is `ast.Attribute` → `core.field` (the op exists today);
- methods are `@overload_method` — ordinary DSL code, erased to free
  functions (which the GPU targets require anyway);
- numpy structured dtypes converge on the same `Record` through the ndarray
  kind — one struct story for both roads.


In [5]:
# The Record machinery, previewed today (only the *decorator sugar* is pending):
Color = T.Record("Color", (("r", T.f32), ("g", T.f32), ("b", T.f32)))
cb = Builder(CORE_OPS)
c = cb.emit("core.env", type=Color, slot=(0,))
g = cb.emit("core.field", c, name="g")
print("c.g :", g.type, " — field access types today; the @record sugar is ch11's")

c.g : f32  — field access types today; the @record sugar is ch11's
